# Aurion — YOLO11 n / s / m

Dataset con split agrupado por imagen original (sin fuga entre particiones).
843 originales → train 590 (1770 img, 3 variantes) / valid 126 / test 127 (1 variante).

Entrena los tres tamaños con hiperparámetros idénticos y saca una tabla de compromiso
precisión/velocidad sobre `test`.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU A100`.
Sin GPU esto pasa de ~2 h a varios días.

## 1. Comprobar GPU

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available())
print("gpu  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")
assert torch.cuda.is_available(), "Sin GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> A100."

## 2. Instalar ultralytics

In [ ]:
%pip install -q ultralytics

import ultralytics
print("ultralytics:", ultralytics.__version__)

## 3. Subir el dataset

Panel de la izquierda → icono de **carpeta** → botón de **subir archivo** (la hoja con
la flecha) → elige `aurion_split.zip`. Se queda en `/content`, que es donde abre.

Sale una rueda de progreso abajo del panel. Con 190 MB tarda un rato: **espera a que
desaparezca** antes de ejecutar la celda. Si lanzas la celda a medias, el zip está
incompleto y falla al descomprimir.

Si Colab te desconecta, el archivo se pierde y hay que resubirlo.

In [ ]:
import pathlib, zipfile

ZIP = pathlib.Path('/content/aurion_split.zip')

if not ZIP.exists():
    otros = list(pathlib.Path('/content').rglob('aurion_split.zip'))
    if otros:
        ZIP = otros[0]
        print(f"Lo he encontrado en {ZIP}")
    else:
        raise FileNotFoundError(
            "No hay aurion_split.zip en /content.\n"
            "Subelo por el panel de la izquierda y espera a que acabe la rueda de progreso."
        )

mb = ZIP.stat().st_size / 1e6
print(f"{ZIP} — {mb:.0f} MB (deberian ser ~190)")

# Un zip truncado revienta ya al abrirlo: el indice va al final del archivo,
# asi que si la subida se corto, no hay indice que leer.
try:
    with zipfile.ZipFile(ZIP) as z:
        if z.testzip() is not None:
            raise ValueError("Hay entradas corruptas dentro del zip.")
        n = len(z.namelist())
except zipfile.BadZipFile:
    raise SystemExit(
        f"\nEl zip esta incompleto: solo {mb:.0f} MB de los ~190 esperados.\n"
        "La subida se corto a medias.\n\n"
        "Que hacer:\n"
        "  1. Borra aurion_split.zip en el panel de la izquierda (clic derecho -> Eliminar).\n"
        "  2. Vuelve a subirlo y NO ejecutes esta celda hasta que la rueda de progreso\n"
        "     del panel haya desaparecido del todo.\n"
        "  3. Si se corta otra vez, prueba con la pestana en primer plano: Colab corta\n"
        "     subidas cuando la pestana pasa a segundo plano mucho rato.\n"
    ) from None

print(f"Zip integro — {n} entradas.")

In [ ]:
import zipfile

DEST = pathlib.Path('/content/aurion_split')

with zipfile.ZipFile(ZIP) as z:
    z.extractall(DEST)

for s in ('train', 'valid', 'test'):
    n_img = len(list((DEST / s / 'images').glob('*')))
    n_lbl = len(list((DEST / s / 'labels').glob('*')))
    print(f"{s:<6}: {n_img:>5} imagenes / {n_lbl:>5} labels")

## 4. Reescribir data.yaml

El `data.yaml` generado en local lleva `path: C:/Users/Marc/Desktop/aurion_split`.
Esa ruta no existe aquí, así que hay que apuntarla a `/content/aurion_split`.
Sin esto el entrenamiento falla con *dataset not found*.

In [ ]:
CLASES = [
    'palet_bueno',
    'palet_roto',
    'paquete_emb_correct_dim_correct',
    'paquete_emb_correct_dim_incorrect',
    'paquete_emb_incorrect_dim_correct',
    'paquete_emb_incorrect_dim_incorrect',
]

YAML = DEST / 'data.yaml'
YAML.write_text(
    f"path: {DEST}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n\n"
    f"nc: {len(CLASES)}\n"
    f"names: {CLASES}\n",
    encoding='utf-8',
)
print(YAML.read_text())

## 5. Entrenar n, s y m

150 épocas cada uno, mismos hiperparámetros salvo el tamaño del modelo. Eso es lo que
hace comparable la tabla: si cambias también el batch o el imgsz, ya no sabes si la
diferencia viene de la arquitectura o del resto.

En A100 esto son del orden de 20 / 30 / 45 min → **1,5-2 h en total**. El `patience=30`
suele cortarlos antes. Cada modelo guarda su run por separado, así que si Colab te
desconecta a mitad no pierdes los que ya acabaron: reejecuta y los saltará.

`cache='ram'` mantiene las 1770 imágenes en memoria (~2 GB); en A100 vas sobrado y
quita el cuello de botella del dataloader. Si por lo que sea petara por RAM, quítalo.

In [ ]:
import time
from ultralytics import YOLO

MODELOS = ['yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt']
RUNS = pathlib.Path('/content/runs')

tiempos = {}

for peso in MODELOS:
    tag = peso.replace('.pt', '')          # yolo11n, yolo11s, yolo11m
    destino = RUNS / tag / 'weights' / 'best.pt'

    if destino.exists():
        print(f"[{tag}] ya entrenado, lo salto.")
        continue

    print(f"\n{'=' * 62}\n[{tag}] entrenando\n{'=' * 62}")
    t0 = time.time()

    YOLO(peso).train(
        data=str(YAML),
        epochs=150,
        imgsz=640,
        patience=30,
        batch=32,
        device=0,
        workers=8,
        cache='ram',
        project=str(RUNS),
        name=tag,
        seed=0,
        exist_ok=True,
    )

    tiempos[tag] = time.time() - t0
    print(f"[{tag}] listo en {tiempos[tag] / 60:.1f} min")

print("\nEntrenamiento terminado.")

## 6. Tabla de compromiso precisión / velocidad

Los tres evaluados sobre `test`, que ninguno ha visto: `valid` se usó para el early
stopping, así que sus métricas ya están sesgadas.

La tabla cruza precisión (mAP) contra coste (parámetros y ms/imagen). Ese cruce es el
resultado: si `m` te da +0.02 de mAP a cambio de 3× el tiempo de inferencia, para una
cinta de paletizado la respuesta razonable es `s` o `n`, y ahora puedes justificarlo.

**La salida de esta celda es la que me pegas.**

In [ ]:
tabla = {}

for peso in MODELOS:
    tag = peso.replace('.pt', '')
    best = RUNS / tag / 'weights' / 'best.pt'
    if not best.exists():
        print(f"[{tag}] sin best.pt, lo salto.")
        continue

    modelo = YOLO(str(best))
    m = modelo.val(data=str(YAML), split='test', imgsz=640, device=0, verbose=False)

    n_par = sum(p.numel() for p in modelo.model.parameters())
    tabla[tag] = {
        'mAP50': m.box.map50,
        'mAP50-95': m.box.map,
        'P': m.box.mp,
        'R': m.box.mr,
        'params_M': n_par / 1e6,
        'ms_img': m.speed['inference'],
        'ap50': m.box.ap50,
        'ap': m.box.ap,
    }

print("\n" + "=" * 78)
print("TEST — 127 imagenes, 127 originales no vistos")
print("=" * 78)
print(f"{'modelo':<10} {'mAP50':>8} {'mAP50-95':>9} {'P':>7} {'R':>7} {'params(M)':>10} {'ms/img':>8}")
for tag, r in tabla.items():
    print(f"{tag:<10} {r['mAP50']:>8.4f} {r['mAP50-95']:>9.4f} {r['P']:>7.4f} "
          f"{r['R']:>7.4f} {r['params_M']:>10.1f} {r['ms_img']:>8.1f}")

print("\n" + "=" * 78)
print("mAP50-95 por clase")
print("=" * 78)
print(f"{'clase':<38} " + ' '.join(f"{t.replace('yolo11', ''):>8}" for t in tabla))
for i, c in enumerate(CLASES):
    print(f"{c:<38} " + ' '.join(f"{r['ap'][i]:>8.4f}" for r in tabla.values()))

if tiempos:
    print("\nTiempo de entrenamiento (min):")
    for tag, s in tiempos.items():
        print(f"  {tag:<10} {s / 60:>6.1f}")

## 7. Curvas y matrices de confusión

Una tanda por modelo. La matriz de confusión es donde vas a ver si las cuatro clases
`paquete_emb_*` se confunden entre ellas, que es el fallo esperable: se diferencian por
dos atributos combinados (embalaje × dimensión) y visualmente se parecen mucho.

In [ ]:
from IPython.display import Image, display, Markdown

for tag in tabla:
    display(Markdown(f"### {tag}"))
    for nombre in ('results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png'):
        p = RUNS / tag / nombre
        if p.exists():
            print(nombre)
            display(Image(filename=str(p), width=820))

## 8. Descargar los pesos

Colab borra `/content` al cerrar la sesión, y aquí no hay Drive donde dejar copia.
**Descarga esto antes de cerrar** o pierdes el entrenamiento.

`best.pt` es lo mínimo. El zip del run entero lleva además las curvas, la matriz de
confusión y `results.csv`, que te harán falta para el informe.

In [ ]:
import shutil
from google.colab import files

# Los tres runs completos: pesos, curvas, matrices, results.csv, args.yaml
shutil.make_archive('/content/aurion_runs', 'zip', str(RUNS))
mb = pathlib.Path('/content/aurion_runs.zip').stat().st_size / 1e6
print(f"aurion_runs.zip — {mb:.0f} MB")

files.download('/content/aurion_runs.zip')